<a href="https://colab.research.google.com/github/DeemonDuck/IPO-Signal-Engine/blob/main/Preprocessing_and_Feature_Engineering_Model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# IPO Prediction Data Pipeline

!pip install yfinance

import yfinance as yf
import pandas as pd
import numpy as np
from google.colab import drive

# 1️⃣ Download Nifty Data

nifty = yf.download("^NSEI", start="2009-01-01", end="2026-01-01")

# FIX: Remove multi-level columns
nifty.columns = nifty.columns.get_level_values(0)

# Reset index so Date becomes a column
nifty = nifty.reset_index()

# Keep required columns
nifty = nifty[["Date", "Close"]]

# Calculate 7-day return
nifty["Nifty_Return_7d"] = nifty["Close"].pct_change(7) * 100

# Keep only feature column
nifty_features = nifty[["Date", "Nifty_Return_7d"]]

# 2️⃣ Load IPO Dataset

excel_path = "/content/drive/MyDrive/IPO_Predictor/ORIGINAL_DATASET/Initial_Public_Offering_from_2010_to_2025.xlsx"

df_raw = pd.read_excel(excel_path)

# Convert Excel → CSV
csv_path = "/content/drive/MyDrive/ipo_dataset.csv"
df_raw.to_csv(csv_path, index=False)

print("Excel converted to CSV")

# Create working dataframe
df = df_raw.copy()

# 3️⃣ Date Processing

df["Date"] = pd.to_datetime(df["Date"])
nifty_features["Date"] = pd.to_datetime(nifty_features["Date"])

# Extract month and year
df["Month"] = df["Date"].dt.month
df["Year"] = df["Date"].dt.year

# 4️⃣ Feature Engineering

df["Demand_Gap"] = df["QIB"] - df["RII"]

df["log_issue_size"] = np.log(df["Issue_Size(crores)"])

# 5️⃣ Create Target Label

df["Apply_Label"] = (df["Listing Gain"] > 10).astype(int)

# 6️⃣ Merge Market Feature

df = pd.merge(df, nifty_features, on="Date", how="left")

# Fill missing values (IPO on weekend)
df["Nifty_Return_7d"] = df["Nifty_Return_7d"].ffill()

# 7️⃣ Remove Leakage Columns

df = df.drop(columns=[
    "IPO_Name",
    "List Price",
    "CMP(BSE)",
    "CMP(NSE)",
    "Current Gains",
    "Listing Gain"
])

# 8️⃣ Drop Date Column

df = df.drop(columns=["Date"])


# 9️⃣ Save Final Dataset

cleaned_path = "/content/drive/MyDrive/ipo_feature_engineered.csv"

df.to_csv(cleaned_path, index=False)

print("Feature engineered dataset saved to Drive")

print("\nPreview:")
print(df.head())

print("\nDataset Shape:", df.shape)

/tmp/ipykernel_166/2647386518.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download("^NSEI", start="2009-01-01", end="2026-01-01")
[*********************100%***********************]  1 of 1 completed


Excel converted to CSV
Feature engineered dataset saved to Drive

Preview:
   Issue_Size(crores)     QIB    HNI    RII   Total  Offer Price  Unnamed: 13  \
0              650.00   36.72  38.24  32.55   36.20          385          NaN   
1              792.00  163.90  57.71  20.28   69.14          150          NaN   
2             4011.60  103.97  34.98   7.73   41.01          800          NaN   
3             1300.00  133.21  72.00  50.87  100.69          675          NaN   
4              254.26    1.30   1.84   2.22    1.87          158          NaN   

   Unnamed: 14  Unnamed: 15  Unnamed: 16  Unnamed: 17  Unnamed: 18  \
0          NaN          NaN          NaN          NaN          NaN   
1          NaN          NaN          NaN          NaN          NaN   
2          NaN          NaN          NaN          NaN          NaN   
3          NaN          NaN          NaN          NaN          NaN   
4          NaN          NaN          NaN          NaN          NaN   

   Unnamed: 19  U

/tmp/ipykernel_166/2647386518.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nifty_features["Date"] = pd.to_datetime(nifty_features["Date"])


In [ ]:
# Path of your saved dataset
file_path = "/content/drive/MyDrive/ipo_feature_engineered.csv"

# Load dataset
df = pd.read_csv(file_path)

# Remove all columns that start with 'Unnamed'
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Remove columns that are completely empty (all NaN)
df = df.dropna(axis=1, how="all")

# Save cleaned dataset
cleaned_path = "/content/drive/MyDrive/ipo_feature_engineered_clean.csv"
df.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved successfully!")

print("\nFinal Columns:")
print(df.columns)

print("\nPreview:")
print(df.head())

print("\nDataset Shape:", df.shape)

Cleaned dataset saved successfully!

Final Columns:
Index(['Issue_Size(crores)', 'QIB', 'HNI', 'RII', 'Total', 'Offer Price',
       'Month', 'Year', 'Demand_Gap', 'log_issue_size', 'Apply_Label',
       'Nifty_Return_7d'],
      dtype='object')

Preview:
   Issue_Size(crores)     QIB    HNI    RII   Total  Offer Price  Month  Year  \
0              650.00   36.72  38.24  32.55   36.20          385      8  2025   
1              792.00  163.90  57.71  20.28   69.14          150      8  2025   
2             4011.60  103.97  34.98   7.73   41.01          800      8  2025   
3             1300.00  133.21  72.00  50.87  100.69          675      8  2025   
4              254.26    1.30   1.84   2.22    1.87          158      8  2025   

   Demand_Gap  log_issue_size  Apply_Label  Nifty_Return_7d  
0        4.17        6.476972            0        -0.432323  
1      143.62        6.674561            1        -0.432323  
2       96.24        8.296945            0        -0.432323  
3       8